In [0]:
%run ./00_configuracion

Configuración cargada correctamente.
Ruta base: /Volumes/workspace/default/tcga_cancer_ml
Número de clases oficiales: 18


Estructura creada correctamente:
/Volumes/workspace/default/tcga_cancer_ml
/Volumes/workspace/default/tcga_cancer_ml/raw
/Volumes/workspace/default/tcga_cancer_ml/trusted
/Volumes/workspace/default/tcga_cancer_ml/refined
/Volumes/workspace/default/tcga_cancer_ml/models
/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq
/Volumes/workspace/default/tcga_cancer_ml/raw/metadata
/Volumes/workspace/default/tcga_cancer_ml/trusted/rnaseq_long
/Volumes/workspace/default/tcga_cancer_ml/trusted/rnaseq_matrix
/Volumes/workspace/default/tcga_cancer_ml/refined/eda_outputs
/Volumes/workspace/default/tcga_cancer_ml/refined/model_metrics
/Volumes/workspace/default/tcga_cancer_ml/refined/predictions
/Volumes/workspace/default/tcga_cancer_ml/refined/visualizations


muestras_unicas_long,filas_samples,muestras_unicas_samples,pacientes_unicos_long,genes_unicos_long
8335,8469,8335,8283,19944


In [0]:
#Importar librerías
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)

print("Rutas del proyecto:")
print("RAW_RNASEQ_PATH:", RAW_RNASEQ_PATH)
print("RAW_METADATA_PATH:", RAW_METADATA_PATH)
print("TRUSTED_LONG_PATH:", TRUSTED_LONG_PATH)

Rutas del proyecto:
RAW_RNASEQ_PATH: /Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq
RAW_METADATA_PATH: /Volumes/workspace/default/tcga_cancer_ml/raw/metadata
TRUSTED_LONG_PATH: /Volumes/workspace/default/tcga_cancer_ml/trusted/rnaseq_long


In [0]:
# Leer metadatos oficiales de las 18 clases
df_meta = spark.read.csv(
    f"{RAW_METADATA_PATH}/metadatos_tcga_oficial_18_clases.csv",
    header=True,
    inferSchema=True
)

print("Columnas de metadatos:")
print(df_meta.columns)

print("Total de registros en metadatos oficiales:")
print(df_meta.count())

display(df_meta.limit(5))

print("Distribución por tipo de cáncer:")
display(
    df_meta
    .groupBy("cancer_type")
    .agg(F.count("*").alias("n_archivos"))
    .orderBy(F.desc("n_archivos"))
)

Columnas de metadatos:
['file_id', 'file_name', 'file_size', 'case_id', 'case_submitter_id', 'sample_id', 'sample_submitter_id', 'sample_type', 'project_id', 'cancer_type', 'cancer_name']
Total de registros en metadatos oficiales:
8469


file_id,file_name,file_size,case_id,case_submitter_id,sample_id,sample_submitter_id,sample_type,project_id,cancer_type,cancer_name
744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.augmented_star_gene_counts.tsv,4231952,878f975b-94fd-4d69-b7e7-1ed3ac2ee438,TCGA-BH-A18H,9321161b-d568-4d38-aed3-956786a06329,TCGA-BH-A18H-01A,Primary Tumor,TCGA-BRCA,BRCA,Breast invasive carcinoma
4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.augmented_star_gene_counts.tsv,4238832,e4fc0909-f284-4471-866d-d8967b6adcbc,TCGA-E2-A14P,58bfe278-a80f-4722-b286-4d966214d244,TCGA-E2-A14P-01A,Primary Tumor,TCGA-BRCA,BRCA,Breast invasive carcinoma
1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.augmented_star_gene_counts.tsv,4239626,e5aae05a-478e-4a55-a27c-12b2b4be302a,TCGA-AN-A04A,1d8f72b7-8596-40e8-afdf-244dd897758c,TCGA-AN-A04A-01A,Primary Tumor,TCGA-BRCA,BRCA,Breast invasive carcinoma
7db46624-6d07-4f49-b0ff-18bb41375de9,b1f7af2a-a39b-44c7-a077-e1df6b18268a.rna_seq.augmented_star_gene_counts.tsv,4236752,7ef7c87e-6f11-4b9b-bb74-3c86036c8d20,TCGA-MH-A855,14f96a1d-5a17-44b1-8c8a-bfabb1413432,TCGA-MH-A855-01A,Primary Tumor,TCGA-KIRP,KIRP,Kidney renal papillary cell carcinoma
25e923e1-24ce-44a9-934c-20df73d33686,acfb6bc0-2972-48f8-9926-5a5ef46e8d86.rna_seq.augmented_star_gene_counts.tsv,4229913,87c5e2cb-0b2e-4f40-927b-5e1ad7de4824,TCGA-P4-A5E6,6906dc9f-4bf1-43ac-9967-1b23f3de1196,TCGA-P4-A5E6-01A,Primary Tumor,TCGA-KIRP,KIRP,Kidney renal papillary cell carcinoma


Distribución por tipo de cáncer:


cancer_type,n_archivos
BRCA,1111
UCEC,553
KIRC,541
LUAD,540
HNSC,520
LGG,516
LUSC,511
THCA,505
PRAD,501
COAD,481


In [0]:
# Esto es importante para que Spark no infiera mal los tipos
# Esquema esperado de los archivos RNA-Seq STAR Counts
schema_rnaseq = StructType([
    StructField("gene_id", StringType(), True),
    StructField("gene_name", StringType(), True),
    StructField("gene_type", StringType(), True),
    StructField("unstranded", DoubleType(), True),
    StructField("stranded_first", DoubleType(), True),
    StructField("stranded_second", DoubleType(), True),
    StructField("tpm_unstranded", DoubleType(), True),
    StructField("fpkm_unstranded", DoubleType(), True),
    StructField("fpkm_uq_unstranded", DoubleType(), True)
])

In [0]:
# Leer archivos RNA-Seq completos desde raw/rnaseq
df_rnaseq_raw = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("recursiveFileLookup", True)
    .schema(schema_rnaseq)
    .csv(RAW_RNASEQ_PATH)
    .withColumn("source_file", F.col("_metadata.file_path"))
)

print("Lectura inicial creada.")

display(df_rnaseq_raw.limit(10))

Lectura inicial creada.


gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded,source_file
gene_id,gene_name,gene_type,null,null,null,null,null,null,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
N_unmapped,null,null,1.1508019E7,1.1508019E7,1.1508019E7,null,null,null,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
N_multimapping,null,null,9559481.0,9559481.0,9559481.0,null,null,null,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
N_noFeature,null,null,6.0322844E7,8.2074872E7,8.1851399E7,null,null,null,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
N_ambiguous,null,null,4268792.0,1216358.0,1231319.0,null,null,null,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
ENSG00000000003.15,TSPAN6,protein_coding,5096.0,2492.0,2604.0,55.2916,30.1217,40.6886,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
ENSG00000000005.6,TNMD,protein_coding,44.0,23.0,21.0,1.4671,0.7993,1.0796,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
ENSG00000000419.13,DPM1,protein_coding,2550.0,1298.0,1252.0,103.9765,56.6443,76.5155,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
ENSG00000000457.14,SCYL3,protein_coding,490.0,521.0,511.0,3.5037,1.9087,2.5783,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
ENSG00000000460.17,C1orf112,protein_coding,434.0,486.0,508.0,3.5778,1.9491,2.6329,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv


In [0]:
# Como cada archivo quedó dentro de una carpeta con su file_id, extraemos ese ID desde la ruta.
# Extraer file_id desde source_file

patron_uuid = r"([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})"

df_rnaseq_raw = df_rnaseq_raw.withColumn(
    "file_id",
    F.regexp_extract(F.col("source_file"), patron_uuid, 1)
)

print("Validación de file_id extraído:")
display(
    df_rnaseq_raw
    .select("source_file", "file_id")
    .distinct()
    .limit(10)
)

Validación de file_id extraído:


source_file,file_id
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/f2f1f9b6-8539-4780-bfc5-42ae7f46a485/d3d30fe1-8e55-436e-85b7-bff934cfcb8a.rna_seq.augmented_star_gene_counts.tsv,f2f1f9b6-8539-4780-bfc5-42ae7f46a485
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b30d8cc1-789d-479d-a624-363012c77639/d4a99056-1263-41dd-910a-a5a34394cf42.rna_seq.augmented_star_gene_counts.tsv,b30d8cc1-789d-479d-a624-363012c77639
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/48875905-17e3-439e-869c-caf83a360d67/391b894d-7639-4e12-97f1-e189233505ff.rna_seq.augmented_star_gene_counts.tsv,48875905-17e3-439e-869c-caf83a360d67
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/9dece4a3-ecef-469d-a882-dfea3e2c61df/161a0325-0f8a-4d41-be34-6740025c4435.rna_seq.augmented_star_gene_counts.tsv,9dece4a3-ecef-469d-a882-dfea3e2c61df
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/19f99f6f-dc62-496c-89b8-a5dbc806eeb5/318db280-feca-4c65-aaad-a4942357507f.rna_seq.augmented_star_gene_counts.tsv,19f99f6f-dc62-496c-89b8-a5dbc806eeb5
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/d28ff63c-3b5c-4918-8ee6-d7c26e3429f6/471bd3b2-2cc3-4c65-9b7b-63979c15ff47.rna_seq.augmented_star_gene_counts.tsv,d28ff63c-3b5c-4918-8ee6-d7c26e3429f6
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/bc5ea28d-d26b-4643-a16f-45b9723d7239/960e2805-9b82-4d03-84fe-39fda23c6646.rna_seq.augmented_star_gene_counts.tsv,bc5ea28d-d26b-4643-a16f-45b9723d7239
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/199a0032-6135-4cfe-8705-81f61d04d061/58cd5fa2-0779-4cbf-8095-d287f21e2d53.rna_seq.augmented_star_gene_counts.tsv,199a0032-6135-4cfe-8705-81f61d04d061
dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/2c752863-dba7-42ec-a427-44692cd6422f/511b6838-b0e6-41c3-8c47-6c0dcbf35393.rna_seq.augmented_star_gene_counts.tsv,2c752863-dba7-42ec-a427-44692cd6422f


In [0]:
# Limpieza de RNA-Seq

df_rnaseq_limpio = (
    df_rnaseq_raw
    # Eliminar filas técnicas N_unmapped, N_multimapping, N_noFeature, etc.
    .filter(~F.col("gene_id").startswith("N_"))

    # Conservar solo genes protein_coding
    .filter(F.col("gene_type") == "protein_coding")

    # Normalizar Ensembl ID quitando versión
    .withColumn(
        "gene_id_base",
        F.split(F.col("gene_id"), "\\.").getItem(0)
    )

    # Reemplazar TPM nulo por 0
    .withColumn(
        "tpm_unstranded",
        F.coalesce(F.col("tpm_unstranded"), F.lit(0.0))
    )

    # Transformación log2(TPM + 1)
    .withColumn(
        "log2_tpm",
        F.log2(F.col("tpm_unstranded") + F.lit(1.0))
    )
)

print("Datos RNA-Seq limpiados.")

display(df_rnaseq_limpio.limit(10))

Datos RNA-Seq limpiados.


gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded,source_file,file_id,gene_id_base,log2_tpm
ENSG00000000003.15,TSPAN6,protein_coding,5096.0,2492.0,2604.0,55.2916,30.1217,40.6886,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000003,5.8148477500083535
ENSG00000000005.6,TNMD,protein_coding,44.0,23.0,21.0,1.4671,0.7993,1.0796,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000005,1.3028161941869545
ENSG00000000419.13,DPM1,protein_coding,2550.0,1298.0,1252.0,103.9765,56.6443,76.5155,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000419,6.7139225926378225
ENSG00000000457.14,SCYL3,protein_coding,490.0,521.0,511.0,3.5037,1.9087,2.5783,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000457,2.171110729965424
ENSG00000000460.17,C1orf112,protein_coding,434.0,486.0,508.0,3.5778,1.9491,2.6329,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000460,2.19465443421053
ENSG00000000938.13,FGR,protein_coding,1172.0,572.0,601.0,17.0552,9.2913,12.5508,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000938,4.17434249619912
ENSG00000000971.16,CFH,protein_coding,290.0,146.0,144.0,1.7885,0.9744,1.3162,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000971,1.4794892709785508
ENSG00000001036.14,FUCA2,protein_coding,4500.0,2594.0,2610.0,78.4798,42.7542,57.7527,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000001036,6.3125163371356
ENSG00000001084.13,GCLC,protein_coding,1175.0,631.0,624.0,6.7102,3.6556,4.938,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000001084,2.9467682836507954
ENSG00000001167.14,NFYA,protein_coding,895.0,485.0,477.0,11.5581,6.2966,8.5055,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000001167,3.6505463005238625


In [0]:
# Unir RNA-Seq limpio con metadatos
columnas_meta = [
    "file_id",
    "file_name",
    "file_size",
    "case_id",
    "case_submitter_id",
    "sample_id",
    "sample_submitter_id",
    "sample_type",
    "project_id",
    "cancer_type",
    "cancer_name"
]

df_meta_sel = df_meta.select(*[c for c in columnas_meta if c in df_meta.columns])

df_trusted_long = (
    df_rnaseq_limpio
    .join(df_meta_sel, on="file_id", how="inner")
)

print("Datos unidos con metadatos.")

display(df_trusted_long.limit(10))

Datos unidos con metadatos.


file_id,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded,source_file,gene_id_base,log2_tpm,file_name,file_size,case_id,case_submitter_id,sample_id,sample_submitter_id,sample_type,project_id,cancer_type,cancer_name
b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000003.15,TSPAN6,protein_coding,5096.0,2492.0,2604.0,55.2916,30.1217,40.6886,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,ENSG00000000003,5.8148477500083535,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma
b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000005.6,TNMD,protein_coding,44.0,23.0,21.0,1.4671,0.7993,1.0796,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,ENSG00000000005,1.3028161941869545,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma
b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000419.13,DPM1,protein_coding,2550.0,1298.0,1252.0,103.9765,56.6443,76.5155,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,ENSG00000000419,6.7139225926378225,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma
b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000457.14,SCYL3,protein_coding,490.0,521.0,511.0,3.5037,1.9087,2.5783,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,ENSG00000000457,2.171110729965424,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma
b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000460.17,C1orf112,protein_coding,434.0,486.0,508.0,3.5778,1.9491,2.6329,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,ENSG00000000460,2.19465443421053,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma
b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000938.13,FGR,protein_coding,1172.0,572.0,601.0,17.0552,9.2913,12.5508,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,ENSG00000000938,4.17434249619912,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma
b41a0bbe-6817-41f5-89e3-9f5321b52eec,ENSG00000000971.16,CFH,protein_coding,290.0,146.0,144.0,1.7885,0.9744,1.3162,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,ENSG00000000971,1.4794892709785508,8bc0c5c2-1e1b-4b

In [0]:
# Crear patient_id y filtrar dataset oficial
df_trusted_long = (
    df_trusted_long
    .withColumn(
        "patient_id",
        F.coalesce(
            F.col("case_submitter_id"),
            F.substring(F.col("sample_submitter_id"), 1, 12),
            F.col("sample_id")
        )
    )
    .filter(F.col("cancer_type").isin(CLASES_PRINCIPALES))
    .filter(F.lower(F.col("sample_type")).contains("primary tumor"))
)

columnas_finales = [
    "file_id",
    "file_name",
    "file_size",
    "case_id",
    "case_submitter_id",
    "sample_id",
    "sample_submitter_id",
    "patient_id",
    "sample_type",
    "project_id",
    "cancer_type",
    "cancer_name",
    "gene_id_base",
    "gene_name",
    "gene_type",
    "tpm_unstranded",
    "log2_tpm",
    "source_file"
]

df_trusted_long = df_trusted_long.select(*columnas_finales)

print("Dataset trusted filtrado.")

display(df_trusted_long.limit(10))

Dataset trusted filtrado.


file_id,file_name,file_size,case_id,case_submitter_id,sample_id,sample_submitter_id,patient_id,sample_type,project_id,cancer_type,cancer_name,gene_id_base,gene_name,gene_type,tpm_unstranded,log2_tpm,source_file
b41a0bbe-6817-41f5-89e3-9f5321b52eec,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,TCGA-HU-A4GD,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma,ENSG00000000003,TSPAN6,protein_coding,55.2916,5.8148477500083535,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
b41a0bbe-6817-41f5-89e3-9f5321b52eec,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,TCGA-HU-A4GD,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma,ENSG00000000005,TNMD,protein_coding,1.4671,1.3028161941869545,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
b41a0bbe-6817-41f5-89e3-9f5321b52eec,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,TCGA-HU-A4GD,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma,ENSG00000000419,DPM1,protein_coding,103.9765,6.7139225926378225,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
b41a0bbe-6817-41f5-89e3-9f5321b52eec,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,TCGA-HU-A4GD,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma,ENSG00000000457,SCYL3,protein_coding,3.5037,2.171110729965424,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
b41a0bbe-6817-41f5-89e3-9f5321b52eec,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,TCGA-HU-A4GD,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma,ENSG00000000460,C1orf112,protein_coding,3.5778,2.19465443421053,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
b41a0bbe-6817-41f5-89e3-9f5321b52eec,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,TCGA-HU-A4GD,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma,ENSG00000000938,FGR,protein_coding,17.0552,4.17434249619912,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
b41a0bbe-6817-41f5-89e3-9f5321b52eec,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa88ce53-8439-40bd-8d93-e1d903d1c7fb,TCGA-HU-A4GD,d455b7cc-07cd-4a4f-80cf-8022870fe994,TCGA-HU-A4GD-01A,TCGA-HU-A4GD,Primary Tumor,TCGA-STAD,STAD,Stomach adenocarcinoma,ENSG00000000971,CFH,protein_coding,1.7885,1.4794892709785508,dbfs:/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq/b41a0bbe-6817-41f5-89e3-9f5321b52eec/8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv
b41a0bbe-6817-41f5-89e3-9f5321b52eec,8bc0c5c2-1e1b-4bfd-952c-f346050f315d.rna_seq.augmented_star_gene_counts.tsv,4291329,aa8

In [0]:
# Validaciones de calidad del dataset trusted

print("Número de muestras únicas:")
display(
    df_trusted_long
    .select("sample_id")
    .distinct()
    .agg(F.count("*").alias("n_muestras_unicas"))
)

print("Número de pacientes únicos:")
display(
    df_trusted_long
    .select("patient_id")
    .distinct()
    .agg(F.count("*").alias("n_pacientes_unicos"))
)

print("Número de genes únicos:")
display(
    df_trusted_long
    .select("gene_id_base")
    .distinct()
    .agg(F.count("*").alias("n_genes_unicos"))
)

print("Distribución por tipo de cáncer:")
display(
    df_trusted_long
    .select("sample_id", "cancer_type")
    .distinct()
    .groupBy("cancer_type")
    .agg(F.count("*").alias("n_muestras"))
    .orderBy(F.desc("n_muestras"))
)

print("Validación de tipos de muestra:")
display(
    df_trusted_long
    .select("sample_id", "sample_type")
    .distinct()
    .groupBy("sample_type")
    .agg(F.count("*").alias("n_muestras"))
)

Número de muestras únicas:


n_muestras_unicas
8335


Número de pacientes únicos:


n_pacientes_unicos
8283


Número de genes únicos:


n_genes_unicos
19944


Distribución por tipo de cáncer:


cancer_type,n_muestras
BRCA,1106
UCEC,549
KIRC,537
LUAD,529
HNSC,520
LGG,516
THCA,505
LUSC,501
PRAD,501
COAD,471


Validación de tipos de muestra:


sample_type,n_muestras
Primary Tumor,8335


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
# Crear tabla candidata de muestras a partir de df_trusted_long


df_samples_candidata = (
    df_trusted_long
    .select(
        "file_id",
        "file_name",
        "file_size",
        "case_id",
        "case_submitter_id",
        "sample_id",
        "sample_submitter_id",
        "patient_id",
        "sample_type",
        "project_id",
        "cancer_type",
        "cancer_name"
    )
    .distinct()
)

print("Filas candidatas en tabla de muestras:")
print(df_samples_candidata.count())

print("Muestras únicas candidatas:")
display(
    df_samples_candidata
    .agg(
        F.count("*").alias("filas_samples_candidata"),
        F.countDistinct("sample_id").alias("muestras_unicas"),
        F.countDistinct("patient_id").alias("pacientes_unicos")
    )
)


Filas candidatas en tabla de muestras:
8469
Muestras únicas candidatas:


filas_samples_candidata,muestras_unicas,pacientes_unicos
8469,8335,8283


In [0]:
# Identificar duplicados por sample_id
duplicados_sample = (
    df_samples_candidata
    .groupBy("sample_id")
    .agg(
        F.count("*").alias("n_filas"),
        F.countDistinct("file_id").alias("n_file_ids"),
        F.countDistinct("patient_id").alias("n_patient_ids"),
        F.first("cancer_type", ignorenulls=True).alias("cancer_type")
    )
    .filter(F.col("n_filas") > 1)
    .orderBy(F.desc("n_filas"))
)

print("Número de sample_id duplicados:")
print(duplicados_sample.count())

display(duplicados_sample.limit(20))

Número de sample_id duplicados:
133


sample_id,n_filas,n_file_ids,n_patient_ids,cancer_type
f26e9844-6044-4b3b-b9df-4f95e14d9e75,3,3,1,GBM
87817041-278c-41d8-9c27-bac80d299903,2,2,1,GBM
e0601d4f-a1df-4f29-baba-f8a67556a822,2,2,1,GBM
3c83ded3-3515-4071-8788-6e8bf2c2a173,2,2,1,GBM
2be9b353-e4bb-4356-b9a1-080be2ebc80d,2,2,1,LUSC
f091c76e-5d01-4f4b-8f6f-463c9f302930,2,2,1,LUSC
ddb395e5-4b12-4610-9755-01e444b35c2e,2,2,1,LUSC
20342f47-d86b-468f-9450-a0fef449e21e,2,2,1,GBM
22be9844-4d9e-4372-a46a-b5c64480e5aa,2,2,1,LUSC
eba3e660-2d59-4604-87f9-f4985c51ce55,2,2,1,GBM


In [0]:

# Seleccionar un único archivo por sample_id

ventana_sample = (
    Window
    .partitionBy("sample_id")
    .orderBy(
        F.col("file_size").desc_nulls_last(),
        F.col("file_id").asc()
    )
)

df_samples_final = (
    df_samples_candidata
    .withColumn("rn", F.row_number().over(ventana_sample))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Tabla de muestras después de deduplicar:")
display(
    df_samples_final
    .agg(
        F.count("*").alias("filas_samples_final"),
        F.countDistinct("sample_id").alias("muestras_unicas_final"),
        F.countDistinct("patient_id").alias("pacientes_unicos_final")
    )
)

print("Distribución final por tipo de cáncer:")
display(
    df_samples_final
    .groupBy("cancer_type")
    .agg(F.count("*").alias("n_muestras"))
    .orderBy(F.desc("n_muestras"))
)

Tabla de muestras después de deduplicar:


filas_samples_final,muestras_unicas_final,pacientes_unicos_final
8335,8335,8283


Distribución final por tipo de cáncer:


cancer_type,n_muestras
BRCA,1106
UCEC,549
KIRC,537
LUAD,529
HNSC,520
LGG,516
THCA,505
PRAD,501
LUSC,501
COAD,471


In [0]:
#Filtrar tabla long usando solo el file_id seleccionado

df_ids_finales = (
    df_samples_final
    .select("sample_id", "file_id")
    .dropDuplicates()
)

df_trusted_long = (
    df_trusted_long
    .join(df_ids_finales, on=["sample_id", "file_id"], how="inner")
)

print("Registros long después de filtrar muestras deduplicadas:")
print(df_trusted_long.count())

Registros long después de filtrar muestras deduplicadas:
166383270


In [0]:

# Garantizar unicidad sample_id + gene_id_base

duplicados_sample_gene = (
    df_trusted_long
    .groupBy("sample_id", "gene_id_base")
    .agg(F.count("*").alias("n"))
    .filter(F.col("n") > 1)
)

print("Duplicados sample_id + gene_id_base antes de agregación:")
print(duplicados_sample_gene.count())

# Agregación defensiva:
# si por alguna razón queda más de una fila por muestra-gen,
# se promedia la expresión.
columnas_muestra_gen = [
    "file_id",
    "file_name",
    "file_size",
    "case_id",
    "case_submitter_id",
    "sample_id",
    "sample_submitter_id",
    "patient_id",
    "sample_type",
    "project_id",
    "cancer_type",
    "cancer_name",
    "gene_id_base"
]

df_trusted_long = (
    df_trusted_long
    .groupBy(*columnas_muestra_gen)
    .agg(
        F.first("gene_name", ignorenulls=True).alias("gene_name"),
        F.first("gene_type", ignorenulls=True).alias("gene_type"),
        F.avg("tpm_unstranded").alias("tpm_unstranded"),
        F.avg("log2_tpm").alias("log2_tpm"),
        F.first("source_file", ignorenulls=True).alias("source_file")
    )
)

print("Registros long después de agregación defensiva:")
print(df_trusted_long.count())

Duplicados sample_id + gene_id_base antes de agregación:
150030
Registros long después de agregación defensiva:
166233240


In [0]:
# Validación final de consistencia


print("Validación final después de deduplicación:")

display(
    df_trusted_long
    .agg(
        F.countDistinct("sample_id").alias("muestras_unicas_long"),
        F.countDistinct("patient_id").alias("pacientes_unicos_long"),
        F.countDistinct("gene_id_base").alias("genes_unicos_long")
    )
)

display(
    df_samples_final
    .agg(
        F.count("*").alias("filas_samples_final"),
        F.countDistinct("sample_id").alias("muestras_unicas_samples"),
        F.countDistinct("patient_id").alias("pacientes_unicos_samples")
    )
)

Validación final después de deduplicación:


muestras_unicas_long,pacientes_unicos_long,genes_unicos_long
8335,8283,19944


filas_samples_final,muestras_unicas_samples,pacientes_unicos_samples
8335,8335,8283


In [0]:
# Guardar dataset limpio en zona trusted como Delta
(
    df_trusted_long
    .repartition("cancer_type")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("cancer_type")
    .save(TRUSTED_LONG_PATH)
)

print("Dataset trusted guardado en:")
print(TRUSTED_LONG_PATH)

Dataset trusted guardado en:
/Volumes/workspace/default/tcga_cancer_ml/trusted/rnaseq_long


In [0]:
# Catalogar tabla trusted para SparkSQL
# Leer desde el Volume y crear tabla managed en Unity Catalog
spark.sql("DROP TABLE IF EXISTS workspace.default.trusted_tcga_rnaseq_long_18_clases")

df_from_volume = spark.read.format("delta").load(TRUSTED_LONG_PATH)

(
    df_from_volume
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("cancer_type")
    .saveAsTable("workspace.default.trusted_tcga_rnaseq_long_18_clases")
)

print("Tabla catalogada:")
print("workspace.default.trusted_tcga_rnaseq_long_18_clases")

print("\nVerificación de registros en tabla catalogada:")
display(
    spark.sql(
        "SELECT cancer_type, COUNT(DISTINCT sample_id) as n_muestras "
        "FROM workspace.default.trusted_tcga_rnaseq_long_18_clases "
        "GROUP BY cancer_type ORDER BY n_muestras DESC"
    )
)

Tabla catalogada:
workspace.default.trusted_tcga_rnaseq_long_18_clases

Verificación de registros en tabla catalogada:


cancer_type,n_muestras
BRCA,1106
UCEC,549
KIRC,537
LUAD,529
HNSC,520
LGG,516
THCA,505
PRAD,501
LUSC,501
COAD,471


In [0]:
# Crear tabla trusted de muestras únicas
df_samples = df_samples_final

# Validación rápida antes de guardar
print("Validación de df_samples antes de guardar:")

display(
    df_samples
    .agg(
        F.count("*").alias("filas_samples"),
        F.countDistinct("sample_id").alias("muestras_unicas"),
        F.countDistinct("patient_id").alias("pacientes_unicos")
    )
)

# Eliminar tabla anterior si existe
spark.sql("DROP TABLE IF EXISTS workspace.default.trusted_tcga_samples_18_clases")

# Guardar tabla trusted de muestras únicas
(
    df_samples
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.trusted_tcga_samples_18_clases")
)

print("Tabla de muestras creada correctamente:")
print("workspace.default.trusted_tcga_samples_18_clases")

# Distribución final por clase
display(
    df_samples
    .groupBy("cancer_type")
    .agg(F.count("*").alias("n_muestras"))
    .orderBy(F.desc("n_muestras"))
)

Validación de df_samples antes de guardar:


filas_samples,muestras_unicas,pacientes_unicos
8335,8335,8283


Tabla de muestras creada correctamente:
workspace.default.trusted_tcga_samples_18_clases


cancer_type,n_muestras
BRCA,1106
UCEC,549
KIRC,537
LUAD,529
HNSC,520
LGG,516
THCA,505
PRAD,501
LUSC,501
COAD,471


In [0]:
# Crear diccionario de genes

df_genes = (
    df_trusted_long
    .select(
        "gene_id_base",
        "gene_name",
        "gene_type"
    )
    .distinct()
)

spark.sql("DROP TABLE IF EXISTS workspace.default.trusted_tcga_gene_dictionary")

(
    df_genes
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.trusted_tcga_gene_dictionary")
)

print("Tabla de diccionario de genes creada:")
print("workspace.default.trusted_tcga_gene_dictionary")
print(f"Total de genes únicos: {df_genes.count()}")

display(df_genes.limit(10))

Tabla de diccionario de genes creada:
workspace.default.trusted_tcga_gene_dictionary
Total de genes únicos: 19944


gene_id_base,gene_name,gene_type
ENSG00000110880,CORO1C,protein_coding
ENSG00000166352,IFTAP,protein_coding
ENSG00000095627,TDRD1,protein_coding
ENSG00000152154,TMEM178A,protein_coding
ENSG00000172689,MS4A10,protein_coding
ENSG00000100485,SOS2,protein_coding
ENSG00000119392,GLE1,protein_coding
ENSG00000122435,TRMT13,protein_coding
ENSG00000196700,ZNF512B,protein_coding
ENSG00000186106,ANKRD46,protein_coding


In [0]:
# Validación final usando SparkSQL

display(
    spark.sql("""
        SELECT cancer_type, COUNT(DISTINCT sample_id) AS n_muestras
        FROM workspace.default.trusted_tcga_rnaseq_long_18_clases
        GROUP BY cancer_type
        ORDER BY n_muestras DESC
    """)
)

display(
    spark.sql("""
        SELECT COUNT(DISTINCT sample_id) AS muestras,
               COUNT(DISTINCT patient_id) AS pacientes,
               COUNT(DISTINCT gene_id_base) AS genes
        FROM workspace.default.trusted_tcga_rnaseq_long_18_clases
    """)
)

cancer_type,n_muestras
BRCA,1106
UCEC,549
KIRC,537
LUAD,529
HNSC,520
LGG,516
THCA,505
PRAD,501
LUSC,501
COAD,471


muestras,pacientes,genes
8335,8283,19944
